# Week 33, Day 2: Short-term depression: the Tsodyks-Markram model

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w033/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: A depressing synapse

Write `tm_train(f, U, tau, n)`, the responses (as U x, with A_max = 1) to n spikes at f Hz, starting from x = 1, and `x_ss(f, U, tau)` from the formula. Compute the steady-state response per spike and per second for U = 0.5 and τ_rec = 800 ms at 1, 5, 20, 40 and 80 Hz.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def tm_train(f, U, tau, n=10):
    dt = 1000.0 / f
    x, out = 1.0, []
    for k in range(n):
        out.append(U * x)
        x = ...                                   # release, then recover over one interval
    return np.array(out)

def x_ss(f, U, tau):
    E = np.exp(-1000.0 / f / tau)
    return ...

per_spike, per_second = {}, {}
for f in (1, 5, 20, 40, 80):
    per_spike[f] = 0.5 * x_ss(f, 0.5, 800.0)
    per_second[f] = f * per_spike[f]
    print(f"{f:3d} Hz: steady response {per_spike[f]:.3f} of the maximum per spike, {per_second[f]:.2f} per second")
print(f"limiting frequency 1/(U tau) = {1000 / (0.5 * 800):.1f} Hz;  1/tau = {1000 / 800:.2f} per second")
plt.bar(range(10), tm_train(20, 0.5, 800.0)); plt.xlabel("spike number"); plt.ylabel("response (U x)"); plt.show()

In [ ]:
import numpy as np
def test_train():
    "tm_train returns U x before each spike, with x updated as in the model"
    a = tm_train(20, 0.5, 800.0, n=3)
    x2 = 1 - (1 - 0.5) * np.exp(-50 / 800)
    assert abs(a[0] - 0.5) < 1e-12 and abs(a[1] - 0.5 * x2) < 1e-12

def test_steady():
    "the tenth-plus responses approach the steady-state formula, and transmitted current per second barely changes from 20 to 40 Hz"
    assert abs(tm_train(20, 0.5, 800.0, n=60)[-1] - 0.5 * x_ss(20, 0.5, 800.0)) < 1e-6
    assert abs(per_second[40] / per_second[20] - 1) < 0.1

test_train()
test_steady()
print("All checks passed")

### Exercise: Steps in rate

Use the averaged model dx/dt = (1 - x)/τ - U x r(t), with r in spikes per ms, and transmitted rate U x r. For steps from 10 to 20 Hz and from 20 to 40 Hz at t = 3 s (U = 0.5, τ = 800 ms, dt = 1 ms), compute the transmitted rate just after the step divided by that just before (`jump`), and the value 2.5 s after the step divided by that before (`settle`).

In [ ]:
def step_response(r1, r2, U=0.5, tau=800.0, T=6000, dt=1.0):
    t = np.arange(0, T, dt)
    r = np.where(t < 3000, r1, r2) / 1000.0
    x = np.empty_like(t); xv = 1.0
    for k in range(len(t)):
        xv += dt * ((1 - xv) / tau - U * xv * r[k])
        x[k] = xv
    return t, U * x * r * 1000

jump, settle = {}, {}
for r1, r2 in ((10, 20), (20, 40)):
    t, out = step_response(r1, r2)
    before = out[t == 2999][0]
    jump[(r1, r2)] = ...
    settle[(r1, r2)] = ...
    print(f"{r1} to {r2} Hz: jumps by {jump[(r1, r2)]:.2f}, settles at {settle[(r1, r2)]:.2f} of the earlier level")
    plt.plot(t / 1000, out, label=f"{r1} to {r2} Hz")
plt.xlabel("s"); plt.ylabel("transmitted (resources per s)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_jump():
    "the transmitted rate jumps by the same factor, 2, for both steps"
    assert abs(jump[(10, 20)] - 2) < 0.05 and abs(jump[(20, 40)] - 2) < 0.05

def test_settle():
    "afterwards both settle to within 15% of the transmission before the step"
    assert all(abs(settle[k] - 1) < 0.15 for k in settle)

test_jump()
test_settle()
print("All checks passed")

### Exercise: Release fraction and recovery from pairs

The array `ppr` holds paired-pulse ratios measured (with noise) at the intervals `gaps`. Fit a straight line to ln(1 - ppr) against the interval and recover `U_fit` and `tau_fit` (ms).

In [ ]:
r = np.random.default_rng(33)
gaps = np.array([20, 50, 100, 200, 400, 700, 1000.0])
ppr = 1 - 0.6 * np.exp(-gaps / 500) + r.normal(0, 0.02, gaps.size)
slope, intercept = np.polyfit(gaps, np.log(1 - ppr), 1)
U_fit = ...
tau_fit = ...
print(f"U = {U_fit:.2f}, tau_rec = {tau_fit:.0f} ms")
plt.plot(gaps, ppr, "o"); g = np.linspace(0, 1000, 200); plt.plot(g, 1 - U_fit * np.exp(-g / tau_fit))
plt.xlabel("interval (ms)"); plt.ylabel("A2 / A1"); plt.show()

In [ ]:
import numpy as np
def test_fit():
    "the fitted U is within 0.1 of 0.6 and the recovery time within 25% of 500 ms"
    assert abs(U_fit - 0.6) < 0.1 and abs(tau_fit - 500) / 500 < 0.25

test_fit()
print("All checks passed")